# 量化
在 Hugging Face `Transformers` 的官方文档与实践中，最常用的三类集成方式是加载 **GPTQ**、**AWQ** 以及 **bitsandbytes（bnb）**。在代码层面，它们通常通过 `AutoModel*.from_pretrained(..., quantization_config=...)` 搭配相应的配置类（如 `GPTQConfig`、`AwqConfig`、`BitsAndBytesConfig`）实现相对统一的调用体验。

如果从使用场景来区分，**GPTQ 和 AWQ** 主要面向**推理部署与加速**，它们属于 PTQ（Post-Training Quantization）算法，生成的模型通常以量化后的检查点形式保存，加载后显存占用低且推理速度快。**bitsandbytes** 则既常用于 8bit/4bit **推理**，也是诸如 QLoRA 在内的一系列**低显存微调方案**的核心依赖，尤其擅长让大模型在单卡上完成 4-bit 训练。

# GPTQ
GPTQ 的量化目标是最小化量化前后激活值的平方误差：

$$ \min_{\widehat{\mathbf{W}}} \| \mathbf{WX} - \widehat{\mathbf{W}}\mathbf{X} \|_2^2 \tag{13.5} $$

其中，$\mathbf{W}$ 为原始权重矩阵，$\mathbf{X}$ 为输入激活值矩阵，$\widehat{\mathbf{W}}$ 为量化后的权重矩阵。
GPTQ 的成功依赖于三个**关键机制**：

（1）**二阶信息补偿**：它利用海森矩阵（Hessian Matrix，$\mathbf{H} = 2\mathbf{X}\mathbf{X}^\top$）的二阶信息来判断权重的重要性。**这就是识别“冗余参数”的重要数学工具**。海森矩阵描述了损失函数曲面的曲率，如果某个权重方向上的曲率很小（平坦），说明该权重的微小变化对总误差影响不大，它是相对“冗余”的；反之则是“关键”参数。GPTQ 利用其逆矩阵 $\mathbf{H}^{-1}$ 来更新剩余权重，以补偿当前权重量化带来的误差 $\delta$：

$$ \boldsymbol{\delta}_F = - \frac{w_q - Q(w_q)}{[\mathbf{H}_F^{-1}]_{qq}} \cdot (\mathbf{H}_F^{-1})_{:, q} \tag{13.6} $$

其中，$w_q$ 是当前被量化的权重，$Q(w_q)$ 是其量化值，$\boldsymbol{\delta}_F$ 是对剩余未量化权重集合 $F$ 的更新向量，$\mathbf{H}_F^{-1}$ 是对应当前未量化权重的海森矩阵逆矩阵。

> 这里给出的只是 GPTQ 在分块/局部参数子集上的近似更新形式，省略了部分实现细节，主要是理解它利用二阶信息做误差补偿的思想。

（2）**任意顺序与延迟批量更新**：GPTQ 发现大模型不需要像 OBQ 那样进行昂贵的“贪心排序”，只需按顺序量化即可。同时，它引入了 **Lazy Batch-Updates（延迟批量更新）** 策略，将计算密集型的更新操作分块执行（如 128 列为一组），大大提升了 GPU 利用率。

（3）**Cholesky 分解**：为了解决大模型下海森矩阵逆计算的数值不稳定性问题，GPTQ 引入了 Cholesky 分解，确保了算法在千亿参数规模下的稳健运行。

# AWQ
权重的“重要性”并不取决于权重本身的大小，而取决于它所处理的激活值的大小。
找出那些对应较大激活值的权重通道，给它们乘上一个放大系数 $s$（Scale up），同时在输入 $x$ 上除以 $s$。
- **原理**：在不改变线性层输出（例如 $y = \mathbf{w}\mathbf{x}$）的前提下，将“重要通道”的权重按系数 $s$ 放大、并将对应输入按 $1/s$ 缩小，使得整体计算在数学上保持等价，但被放大的权重在量化时的相对误差更小。
- **效果**：当权重被放大后，其数值范围变大，相对量化误差（Relative Quantization Error）就会变小。AWQ 的优化目标是找到一组最优的缩放因子 $\mathbf{s}$，使得量化误差最小：

  $$ \mathbf{s}^* = \arg \min_{\mathbf{s}} \mathcal{L}(\mathbf{s}) \tag{13.7} $$

  $$ \mathcal{L}(\mathbf{s}) = \| Q(\mathbf{W} \cdot \text{diag}(\mathbf{s})) (\text{diag}(\mathbf{s})^{-1} \cdot \mathbf{X}) - \mathbf{WX} \| \tag{13.8} $$

  其中，$Q(\cdot)$ 表示量化函数，$\mathbf{W}$ 为原始权重，$\mathbf{X}$ 为输入特征，$\mathbf{s}$ 为我们需要寻找的最佳缩放因子向量，$\text{diag}(\mathbf{s})$ 是由 $\mathbf{s}$ 构成的对角矩阵。这就好比用一把尺子去量物体，把物体放大后再量，读数的相对精度自然就高了。最终 AWQ 在全 INT 量化下也能达到与混合精度相当的性能（PPL 13.0）。


# BNB
**LLM.int8()** 解决这一问题的主要原理是**混合精度分解（Mixed-precision Decomposition）**。它就像一个智能筛子，在推理过程中动态探测特征值的大小。对于 99.9% 的常规数值，使用**向量级量化（Vector-wise Quantization）**把它们压缩为 8-bit 进行矩阵乘法，以节省显存：

$$ \mathbf{C}_{f16} \approx \frac{1}{\mathbf{c}_{x} \otimes \mathbf{c}_{w}} \mathbf{C}_{i32} \tag{13.9} $$

其中，$\mathbf{C}_{f16}$ 是近似的 FP16 输出结果，$\mathbf{C}_{i32}$ 是 INT8 矩阵乘法得到的 INT32 结果，$\mathbf{c}_{x}$ 和 $\mathbf{c}_{w}$ 分别是输入 $\mathbf{X}$ 和权重 $\mathbf{W}$ 的量化缩放因子（Scaling Factors）。

而对于那 0.1% 超过阈值（如 6.0）的“离群”维度 $O$，则自动拆分出来，保持 **FP16** 高精度计算。最后将两部分结果合并（见图 13-5）：

$$ \mathbf{C}_{f16} \approx \underbrace{\sum_{h \in O} \mathbf{X}_{f16}^h \mathbf{W}_{f16}^h}_{\text{离群值 (FP16)}} + \underbrace{S \cdot \sum_{h \notin O} \mathbf{X}_{i8}^h \mathbf{W}_{i8}^h}_{\text{常规值 (INT8)}} \tag{13.10} $$

其中，$h$ 代表特征维度，$O$ 是离群特征维度的集合，$S$ 是去归一化项（对应上面的缩放因子乘积）。为了便于理解，这里对缩放因子 $\mathbf{c}_{x}$、$\mathbf{c}_{w}$ 和 $S$ 的张量维度及广播方式做了简化，实际工程实现中会针对 batch/head/channel 等维度分别维护缩放因子。

这种“抓大放小”的策略，让我们可以在几乎不损失任何精度的情况下，用 INT8 的显存开销运行超大模型。


In [ ]:
import torch
import ssl
import os

os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
os.environ["HF_DATASETS_ENDPOINT"] = "https://hf-mirror.com"
os.environ["CURL_CA_BUNDLE"] = ""
ssl._create_default_https_context = ssl._create_unverified_context

from transformers import AutoModelForCausalLM, AutoTokenizer, GPTQConfig
from datasets import load_dataset

model_id = "Qwen/Qwen2.5-1.5B-Instruct"
device = "cuda" if torch.cuda.is_available() else "cpu"

# 加载校准数据
calib_dataset = load_dataset("wikitext", "wikitext-2-raw-v1", split="train")
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

num_samples = 128
calibration_data = [
    d["text"] for d in calib_dataset.shuffle().select(range(num_samples)) if d["text"].strip()
]

gptq_config = GPTQConfig(
    bits=4,
    dataset=calibration_data,
    tokenizer=tokenizer,
)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=gptq_config,
    device_map="auto",
    trust_remote_code=True,
)

print(model)

ImportError: Loading a GPTQ quantized model requires optimum (`pip install optimum`)

In [ ]:
def get_version(package_name):
    try:
        return importlib.metadata.version(package_name)
    except Exception:
        return "not found"

import importlib.metadata
print("transformers:", get_version("transformers"))
print("optimum:", get_version("optimum"))

transformers: 4.57.3
optimum: 2.1.0
